# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arhxmz/Flyrank-Internship-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane maps to classification, deployed as a ranking.

At the modeling level, this is classification: I'm predicting whether a page is declining (trend_direction == "down") from its trailing-90-day signals — not clustering (I have a real label to check against, not just groups to find), and not regression (the outcome is a state, not a number to hit exactly).

But editors don't have time to act on a flat list of 16,000 flagged pages — they can fix a handful this week. So the model's predicted probability of decline becomes the sort key: a ranked queue, worked top-down. That's the classification-as-ranking framing — a classifier trained on an observed label, whose output decides which page gets fixed first.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining = (trend_direction == "down") — a proxy, not a clean observed outcome.

trend_pct is a real measurement (actual click/impression change over 90 days); trend_direction buckets it into a label using a threshold, so it's part-observed, part-defined. That means trend_direction/trend_pct are the label only — never features, or the model just re-derives itself.

It's also a proxy in time: this captures decline already observed in the past 90 days, not a predicted future drop. A stricter version would predict a future window from past-only features — the starter data doesn't split that way yet.

In [5]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
df["is_declining"].value_counts(normalize=True)

is_declining
1    0.542067
0    0.457933
Name: proportion, dtype: float64

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@K.

Of the top K pages the model ranks highest, what fraction are actually declining. I'm not using plain accuracy, because the base rate of decline is already 54.2% — a model that guesses "down" for every page gets 54% accuracy for free and tells me nothing about whether the ranking is any good.

K matters because editors don't have time to work through every flagged page — they can act on a handful a week. So what I actually care about is whether the top of the queue (e.g. the top 10%, roughly 3,000 pages) is genuinely worth prioritizing, not whether the model is right across all 30,000 rows.

This is computable today as a baseline: the 54.2% base rate is my floor. Any model I train has to beat that at the top of the ranking, not just overall, or it isn't adding value over guessing

In [6]:
base_rate = df["is_declining"].mean()
print(f"Base rate of decline: {base_rate:.1%}")

Base rate of decline: 54.2%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one published content item, for one client, over a trailing-90-day window.

Each row is a single page (content_id) belonging to a single client (client_id), with performance signals aggregated over the last 90 days — not a query, not a session, not a daily snapshot.

In [3]:
import sys
print(sys.executable)

c:\Users\arham\AppData\Local\Programs\Python\Python313\python.exe


In [7]:
lane_cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "days_since_last_update", "avg_position", "ctr", "engagement_rate",
    "impressions_90d", "clicks_90d", "trend_direction", "trend_pct"
]
df[lane_cols].head()

,content_id,client_id,content_type,content_age_days,days_since_last_update,avg_position,ctr,engagement_rate,impressions_90d,clicks_90d,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,10.6,0.76,5.88,3803,29,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,20.3,0.05,0.00,15320,7,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,36.5,0.09,0.00,12581,11,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,6.2,0.49,1.28,11751,58,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,44.0,0.13,0.00,19140,24,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule already exists — and already runs out.

FlyRank already ships hand-written rules in production (health_score, quick-win flags): if-this-then-that thresholds. They work, so I'm not inventing decline-detection from scratch — I'm trying to beat that rule.

The rule runs out because there are 40+ interacting signals per page (position, CTR, engagement, freshness, age, AI traffic share). No one hand-writes an if-statement with that many branches. And the relationships likely aren't consistent — what predicts decline for one content type or client probably differs for another, while a single hardcoded threshold applies the same cutoff everywhere.

ML earns its place here specifically because the pattern is real but too tangled to hand-code — not because ML is automatically better than a rule.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df.groupby("trend_direction")[["avg_position", "ctr", "engagement_rate", "content_age_days"]].mean()

,avg_position,ctr,engagement_rate,content_age_days
trend_direction,,,,
down,15.936305,0.324138,2.437193,236.178637
flat,11.102431,1.376753,1.374514,245.889757
new,9.912433,1.296521,2.135653,238.718247
stable,16.332623,0.517321,2.838801,295.439953
up,22.512739,0.565533,2.989578,288.478806


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.